In [0]:
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown(
    "schema",
    "migration_dev",
    ["migration_dev", "migration_test", "migration_prod_sim"],
)
dbutils.widgets.text("batch_id", "batch_001")

catalog_name = dbutils.widgets.get("catalog").strip()
schema_name = dbutils.widgets.get("schema").strip()
batch_id = dbutils.widgets.get("batch_id").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog_name):
    raise ValueError("Invalid catalog name")

if schema_name not in {
    "migration_dev", "migration_test", "migration_prod_sim"
}:
    raise ValueError("Invalid environment schema")

if not re.fullmatch(r"batch_[0-9]+", batch_id):
    raise ValueError("Invalid batch ID")

table_prefix = f"{catalog_name}.{schema_name}"
volume_root = f"/Volumes/{catalog_name}/{schema_name}"

print(f"Environment: {table_prefix}")
print(f"Batch: {batch_id}")
print(f"Volume root: {volume_root}")    

In [0]:
import hashlib
from pathlib import Path
from pyspark.sql import functions as F

account_number = "TEST00000001"

matches = (
    spark.table(f"{table_prefix}.document_registry")
    .filter(
        (F.col("account_number") == account_number)
        & (F.col("document_type") == "ACCOUNT_STATEMENT")
        & (F.col("statement_date") == F.lit("2026-09-30").cast("date"))
        & (F.col("batch_id") == batch_id)
    )
    .collect()
)

if len(matches) != 1:
    raise RuntimeError(f"Expected one document; found {len(matches)}")

document = matches[0]
archive_path = Path(document["archive_path"])

# Restrict retrieval to the expected archive folder.
allowed_folder = Path(
    f"{volume_root}/archive/{batch_id}"
).resolve()

if archive_path.resolve().parent != allowed_folder:
    raise RuntimeError("Document path is outside the allowed archive folder")

content = archive_path.read_bytes()
actual_hash = hashlib.sha256(content).hexdigest()

if actual_hash != document["document_sha256"]:
    raise RuntimeError("Archive checksum mismatch; retrieval blocked")

print(f"Document ID: {document['document_id']}")
print(f"Account: {document['account_number']}")
print(f"Archive path: {archive_path}")
print(f"Retrieved bytes: {len(content)}")
print("PASS: Retrieved document matches the registry checksum")